# 🌊 High-Reynolds-Number Cavity Flow PINN Formulation Benchmark
### Complete Autonomous Execution Pipeline on Kaggle GPU

This notebook trains and benchmarks streamfunction-pressure ($\psi-p$) vs. streamfunction-vorticity ($\psi-\omega$) PINN formulations on $\mathrm{Re}=1000$ lid-driven cavity flow, verifies physical conditioning, evaluates accuracy against Ghia et al. (1982), and exports all 7 publication figures, JSON metrics, CSVs, and a downloadable `results_bundle.zip`.

--- 
## 1. Hardware & Environment Verification
Verify CUDA acceleration and GPU memory allocation.

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA not detected! Please enable GPU in Kaggle settings: Notebook options -> Accelerator -> GPU T4 x2 or P100.")

!nvidia-smi

--- 
## 2. Clone Repository & Setup Environment
Clones the verified repository directly from GitHub with high-resolution converged ground-truth datasets ($251 \times 251$).

In [ ]:
import os
%cd /kaggle/working

# Clean existing clone if rerun
if os.path.exists('pinn-fluid-formulations'):
    !rm -rf pinn-fluid-formulations

# Clone latest main branch
!git clone https://github.com/KartikeyaGangwar/pinn-fluid-formulations.git
%cd pinn-fluid-formulations

# Install requirements
!pip install -q -r requirements.txt
print("\n[SETUP COMPLETE] Working directory set to:", os.getcwd())

--- 
## 3. Verify Ground Truth Datasets
Ensure all 5 Reynolds number reference solutions ($\mathrm{Re} \in [50, 100, 400, 600, 1000]$) are loaded and verified.

In [ ]:
import os, pickle, glob
import numpy as np

gt_files = sorted(glob.glob('data/gt_data_Re*.pkl'))
print(f"Found {len(gt_files)} Ground-Truth Reference Files:")
for fpath in gt_files:
    with open(fpath, 'rb') as f:
        d = pickle.load(f)
    re = d['parameters']['Re']
    n = d['parameters']['N']
    psi_min = float(d['fields']['psi'].min())
    u_top = d['fields']['u'][-1, :]
    x = d['coordinates']['x']
    u_expected = 16.0 * (x**2) * ((1.0 - x)**2)
    top_err = np.max(np.abs(u_top - u_expected))
    print(f"  • Re={re:4d} | Grid: {n}x{n} | psi_min: {psi_min:.6f} | Top-Wall Error: {top_err:.2e}")

assert len(gt_files) == 5, "Missing GT datasets! Please verify clone."

--- 
## 4. Run Complete PINN Training Suite (8000 Epochs)
Trains both $\psi-p$ and $\psi-\omega$ models with the 3-phase curriculum:
- **Phase 1 (Epochs 0–2000)**: Boundary & Data Anchoring ($\lambda_{\mathrm{PDE}}=0.01, \lambda_{\mathrm{GT}}=10.0$)
- **Phase 2 (Epochs 2000–6000)**: Physics Injection ($\lambda_{\mathrm{PDE}}=0.05, \lambda_{\mathrm{GT}}=5.0$)
- **Phase 3 (Epochs 6000–8000)**: Gradient Sharpening & Cosine LR Decay ($\lambda_{\mathrm{PDE}}=0.30, \lambda_{\mathrm{GT}}=2.0$)

In [ ]:
# -------------------------------------------------------------------------
# COMPLETE 5-REYNOLDS-NUMBER TRAINING SUITE (Re = 50, 100, 400, 600, 1000)
# Total runtime on Kaggle GPU: ~22-25 minutes
# -------------------------------------------------------------------------
# 1. Re=50 (4,000 epochs, ~3 mins)
!python train_all.py --re 50 --epochs 4000 --lr 1e-3

# 2. Re=100 (4,000 epochs, ~3 mins)
!python train_all.py --re 100 --epochs 4000 --lr 1e-3

# 3. Re=400 (4,000 epochs, ~3 mins)
!python train_all.py --re 400 --epochs 4000 --lr 1e-3

# 4. Re=600 (6,000 epochs, ~4 mins)
!python train_all.py --re 600 --epochs 6000 --lr 1e-3

# 5. Re=1000 (8,000 epochs, ~10 mins - Hero benchmark)
!python train_all.py --re 1000 --epochs 8000 --lr 1e-3


--- 
## 5. Run Full Evaluation, Benchmarks & Export Suite
Generates all 7 publication figures, calculates vortex core coordinates, centerline errors vs Ghia (1982), kinetic energy, and enstrophy, and creates `results_bundle.zip`.

In [ ]:
!python -m src.evaluate_and_plot

--- 
## 6. Quantitative Results & Comparison Summary

In [ ]:
import json, pandas as pd

with open('results/metrics_summary.json', 'r') as f:
    metrics = json.load(f)

print("="*75)
print("           QUANTITATIVE BENCHMARK SUMMARY (Re=1000)")
print("="*75)

# Vortex Centers
print(f"\n[VORTEX CORE LOCATIONS]")
print(f"  Ghia et al. (1982) : (0.5313, 0.5625)")
print(f"  Reference FDM      : ({metrics['vortex_centers']['fdm']['primary']['x']:.4f}, {metrics['vortex_centers']['fdm']['primary']['y']:.4f})  | psi_min = {metrics['vortex_centers']['fdm']['primary']['psi_min']:.6f}")
print(f"  psi-p PINN         : ({metrics['vortex_centers']['psi_p']['primary']['x']:.4f}, {metrics['vortex_centers']['psi_p']['primary']['y']:.4f})  | psi_min = {metrics['vortex_centers']['psi_p']['primary']['psi_min']:.6f}")
print(f"  psi-omega PINN     : ({metrics['vortex_centers']['psi_omega']['primary']['x']:.4f}, {metrics['vortex_centers']['psi_omega']['primary']['y']:.4f})  | psi_min = {metrics['vortex_centers']['psi_omega']['primary']['psi_min']:.6f}")

# Centerline Velocity Errors vs Ghia (1982)
print(f"\n[CENTERLINE ERRORS vs GHIA BENCHMARK]")
print(f"  Reference FDM  : u MAE = {metrics['centerline_errors']['fdm']['u_mae']:.4f} | v MAE = {metrics['centerline_errors']['fdm']['v_mae']:.4f}")
print(f"  psi-p PINN     : u MAE = {metrics['centerline_errors']['psi_p']['u_mae']:.4f} | v MAE = {metrics['centerline_errors']['psi_p']['v_mae']:.4f}")
print(f"  psi-omega PINN : u MAE = {metrics['centerline_errors']['psi_omega']['u_mae']:.4f} | v MAE = {metrics['centerline_errors']['psi_omega']['v_mae']:.4f}")

# Energetics
print(f"\n[INTEGRATED DOMAIN ENERGETICS]")
print(f"  Reference FDM  : Kinetic Energy = {metrics['energetics']['fdm']['kinetic_energy']:.6f} | Enstrophy = {metrics['energetics']['fdm']['enstrophy']:.4f}")
print(f"  psi-p PINN     : Kinetic Energy = {metrics['energetics']['psi_p']['kinetic_energy']:.6f} | Enstrophy = {metrics['energetics']['psi_p']['enstrophy']:.4f}")
print(f"  psi-omega PINN : Kinetic Energy = {metrics['energetics']['psi_omega']['kinetic_energy']:.6f} | Enstrophy = {metrics['energetics']['psi_omega']['enstrophy']:.4f}")
print("="*75)

--- 
## 7. Visual Verification: Display All 7 Publication Figures

In [ ]:
from IPython.display import display, Image

figures = [
    ('fig1_schematic_cavity.png', 'Figure 1: Problem Geometry & Regularized Boundary Conditions'),
    ('fig2_loss_convergence.png', 'Figure 2: Multi-Phase Loss & Gradient Convergence Dynamics'),
    ('fig3_flow_topologies.png', 'Figure 3: 2D Flow Topologies (Streamlines, Vorticity, Velocity)'),
    ('fig4_centerline_profiles.png', 'Figure 4: Centerline Velocity Profiles vs Ghia et al. (1982)'),
    ('fig5_nearwall_vorticity_profiles.png', 'Figure 5: Near-Wall Vorticity Gradient & Boundary Shear Zoom'),
    ('fig6_reynolds_sweep.png', 'Figure 6: Multi-Reynolds Topology Evolution (Re=50 -> 1000)'),
    ('fig7_datasparsity_ablation.png', 'Figure 7: GT Data Supervision Sparsity Ablation')
]

for fname, title in figures:
    path = os.path.join('figures', fname)
    if os.path.exists(path):
        print(f"\n{'='*70}\n{title}\n{'='*70}")
        display(Image(filename=path, width=850))

--- 
## 8. Export & Download Archive
Copies `results_bundle.zip` and all generated figures to `/kaggle/working/` so they appear directly in the Kaggle right-sidebar file explorer for 1-click download.

In [ ]:
import shutil

# Copy to top-level /kaggle/working directory for easy download
shutil.copy('results_bundle.zip', '/kaggle/working/results_bundle.zip')
!cp -r figures /kaggle/working/
!cp -r results /kaggle/working/

print("[SUCCESS] All deliverables copied to /kaggle/working/:")
print("  1. /kaggle/working/results_bundle.zip (Download this for paper figures, tables & CSVs!)")
print("  2. /kaggle/working/figures/ (All PNG & PDF publication figures)")
print("  3. /kaggle/working/results/ (JSON metrics, CSVs & LaTeX tables)")

from IPython.display import FileLink
display(FileLink('/kaggle/working/results_bundle.zip'))